# Can we predict late delivery?
Protocol: `protocol.yaml` (temporal holdout from `holdout.csv`, 5-fold walk-forward CV on the training rows, choice by mean CV ROC AUC, holdout scored once).

**Feature decision:** `claim_paid_eur` is compensation paid on a claim *settled for the consignment*, so it is only known after delivery (about 90% of rows with a claim are late). It is not available at dispatch time and is excluded as target leakage. `consignment` (identifier) and `dispatched_on` (split column) are excluded per the protocol.

In [ ]:
import json, hashlib
import numpy as np, pandas as pd
from sklearn.base import clone
from sklearn.model_selection import TimeSeriesSplit
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score

df = pd.read_csv('data.csv')
assert hashlib.sha256(open('data.csv','rb').read()).hexdigest()=='859a474044a292565d72b06a6180ade1c1adbc8c72a47c4a518186876f581f27'
hold = pd.read_csv('holdout.csv')
print(hold.columns.tolist(), len(hold), len(df))

In [ ]:
hold_idx = hold.iloc[:,0].to_numpy()          # 0-based row positions in data.csv
df['_t'] = pd.to_datetime(df['dispatched_on'])
train = df.drop(index=hold_idx).sort_values('_t', kind='stable')   # chronological for walk-forward
test = df.loc[np.sort(hold_idx)]
print(len(train), len(test), train._t.max(), test._t.min())
TARGET='late_delivery'
NUM=['distance_km','weight_kg','parcels']; CAT=['carrier']
FEATURES=NUM+CAT
print(train[TARGET].mean(), test[TARGET].mean())

In [ ]:
def prep(scale=True):
    return ColumnTransformer([('num', StandardScaler() if scale else 'passthrough', NUM),
                              ('cat', OneHotEncoder(handle_unknown='ignore'), CAT)])
# All candidates are named before any is scored
candidates = {
 'baseline_prior': Pipeline([('p',prep()),('m',DummyClassifier(strategy='prior'))]),
 'logreg_C0.1': Pipeline([('p',prep()),('m',LogisticRegression(C=0.1,max_iter=1000))]),
 'logreg_C1': Pipeline([('p',prep()),('m',LogisticRegression(C=1.0,max_iter=1000))]),
 'rf_depth6': Pipeline([('p',prep(False)),('m',RandomForestClassifier(n_estimators=300,max_depth=6,min_samples_leaf=20,random_state=0))]),
 'hgb_small': Pipeline([('p',prep(False)),('m',HistGradientBoostingClassifier(max_depth=3,learning_rate=0.05,max_iter=200,random_state=0))]),
}
print(list(candidates))

In [ ]:
tscv = TimeSeriesSplit(n_splits=5)
Xtr, ytr = train[FEATURES], train[TARGET].to_numpy()
folds = list(tscv.split(Xtr))
cv_folds = [[[int(i) for i in train.index[a]], [int(i) for i in train.index[b]]] for a,b in folds]   # [train, test] row positions
cv_scores = {}
for name, est in candidates.items():
    s=[]
    for a,b in folds:   # preprocessing lives in the pipeline, so it is fitted inside each fold
        m = clone(est).fit(Xtr.iloc[a], ytr[a])
        s.append(float(roc_auc_score(ytr[b], m.predict_proba(Xtr.iloc[b])[:,1])))
    cv_scores[name]=s
cv = pd.DataFrame({k:{'mean':np.mean(v),'std':np.std(v)} for k,v in cv_scores.items()}).T
cv.sort_values('mean',ascending=False)

In [ ]:
chosen = max(cv_scores, key=lambda k: np.mean(cv_scores[k]))
final = clone(candidates[chosen]).fit(Xtr, ytr)
holdout_auc = float(roc_auc_score(test[TARGET], final.predict_proba(test[FEATURES])[:,1]))
print('chosen:', chosen, '| CV mean', np.mean(cv_scores[chosen]), '| holdout ROC AUC (scored once):', holdout_auc)

In [ ]:
evidence = {
 'features': FEATURES,
 'candidates': list(candidates),
 'cv_folds': cv_folds,
 'cv_scores': cv_scores,
 'chosen': chosen,
 'holdout_rows': [int(i) for i in np.sort(hold_idx)],
 'holdout_scores': {chosen: holdout_auc},   # roc_auc, chosen candidate only, scored once
}
json.dump(evidence, open('evidence.json','w'))